In [2]:
import pandas as pd

out = './mbdump_small/'

real = pd.read_csv(f'{out}listening_history_real.tsv', sep='\t')
rec  = pd.read_csv(f'{out}recording.tsv', sep='\t', header=None,
       names=['id','gid','name','artist_credit','length','comment',
              'edits_pending','last_updated','video'])

print(f"real listens         → {len(real):,}")
print(f"real with mbid       → {real['recording_mbid'].notna().sum():,}")
print(f"recording in catalog → {len(rec):,}")

mbids_in_real    = set(real['recording_mbid'].dropna())
mbids_in_catalog = set(rec['gid'].dropna())
overlap = mbids_in_real & mbids_in_catalog
print(f"mbid overlap         → {len(overlap):,}")

# Compare both input formats before generating synthetic tables.
print("\nSample real mbids:")
print(list(mbids_in_real)[:3])
print("\nSample catalog gids:")
print(list(mbids_in_catalog)[:3])

real listens         → 3,743,372
real with mbid       → 19,321
recording in catalog → 79,359
mbid overlap         → 13,215

Sample real mbids:
['9489e656-dc00-469f-9303-4958ca9f895d', '7e8f0b37-e861-4174-958a-486cb8d06309', 'b29a68c8-e4f4-4d43-9b84-a5d225e3a879']

Sample catalog gids:
['1d5cb97b-97cf-4ec0-bc09-ecfb9ee388e3', '9db45b6a-31fe-41f9-bdfa-7fbcd19aa680', 'e4691ae2-6e9b-4816-98ab-b19db2f909b4']


In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

out  = './mbdump_small/'
OPTS = dict(engine='python', on_bad_lines='skip', quoting=3)

random.seed(42)
np.random.seed(42)

# Load MusicBrainz catalog tables used by the generator.
recording = pd.read_csv(f'{out}recording.tsv', sep='\t', header=None,
    names=['id','gid','name','artist_credit','length','comment',
           'edits_pending','last_updated','video'], **OPTS)

artist = pd.read_csv(f'{out}artist.tsv', sep='\t', header=None,
    names=['id','gid','name','sort_name','begin_date_year','begin_date_month',
           'begin_date_day','end_date_year','end_date_month','end_date_day',
           'type','area','gender','comment','edits_pending','last_updated',
           'ended','begin_area','end_area'], **OPTS)

artist_credit = pd.read_csv(f'{out}artist_credit.tsv', sep='\t', header=None,
    names=['id','name','artist_count','ref_count','created','edits_pending','gid'], **OPTS)

artist_credit_name = pd.read_csv(f'{out}artist_credit_name.tsv', sep='\t', header=None,
    names=['artist_credit','position','artist','name','join_phrase'], **OPTS)

release = pd.read_csv(f'{out}release.tsv', sep='\t', header=None,
    names=['id','gid','name','artist_credit','release_group','status',
           'packaging','language','script','barcode','comment',
           'edits_pending','quality','last_updated'], **OPTS)

artist_ids_list = list(artist['id'])

# 1. Build listening history from ListenBrainz.
print("Loading real listens...")
real = pd.read_csv(f'{out}listening_history_real.tsv', sep='\t')
raw_listen_count = len(real)

# Map all top ListenBrainz users before matching so users without MBIDs do not vanish.
real['listen_row_id'] = np.arange(len(real))
lb_users = real['user_id'].dropna().unique()
N_USERS = len(lb_users)
uid_map = {lb: i + 1 for i, lb in enumerate(lb_users)}
real['user_id'] = real['user_id'].map(uid_map)
real['duration_ms'] = pd.to_numeric(real['duration_ms'], errors='coerce').replace(0, np.nan)
user_lookup = real.drop_duplicates('user_id')[['user_id', 'user_name']].copy()

def norm_text(series):
    return (series.fillna('').astype(str)
        .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii')
        .str.lower().str.replace(r'[^a-z0-9]+', ' ', regex=True).str.strip())

recording['id'] = pd.to_numeric(recording['id'], errors='coerce')
recording['artist_credit'] = pd.to_numeric(recording['artist_credit'], errors='coerce')
recording['length'] = pd.to_numeric(recording['length'], errors='coerce')
artist_credit['artist_credit'] = pd.to_numeric(artist_credit['id'], errors='coerce')

recording_lookup = (recording[['id', 'gid', 'length']]
    .rename(columns={'id': 'recording_id', 'length': 'recording_length_ms'})
    .dropna(subset=['recording_id', 'gid']))

# First pass: exact MBID match. This is highest confidence.
mbid_matches = (real[real['recording_mbid'].notna()]
    .merge(recording_lookup, left_on='recording_mbid', right_on='gid', how='inner'))
mbid_matches['match_method'] = 'mbid'
mbid_matches['match_confidence'] = 1.0
matched_rows = set(mbid_matches['listen_row_id'])

# Second pass: exact normalized title + artist credit, only when the catalog key is unique.
catalog = (recording[['id', 'name', 'artist_credit', 'length']]
    .rename(columns={'id': 'recording_id', 'name': 'catalog_track', 'length': 'recording_length_ms'})
    .merge(artist_credit[['artist_credit', 'name']].rename(columns={'name': 'catalog_artist'}),
           on='artist_credit', how='left'))
catalog['track_key'] = norm_text(catalog['catalog_track'])
catalog['artist_key'] = norm_text(catalog['catalog_artist'])
catalog = catalog[(catalog['track_key'] != '') & (catalog['artist_key'] != '')]
catalog['key_count'] = catalog.groupby(['track_key', 'artist_key'])['recording_id'].transform('count')
catalog = catalog[catalog['key_count'] == 1]

remaining = real[~real['listen_row_id'].isin(matched_rows)].copy()
remaining['track_key'] = norm_text(remaining['track_name'])
remaining['artist_key'] = norm_text(remaining['artist_name'])
name_matches = remaining.merge(
    catalog[['recording_id', 'recording_length_ms', 'track_key', 'artist_key']],
    on=['track_key', 'artist_key'], how='inner'
)
duration_gap = (name_matches['duration_ms'] - name_matches['recording_length_ms']).abs()
duration_limit = np.maximum(30_000, name_matches['recording_length_ms'] * 0.10)
duration_ok = name_matches['duration_ms'].isna() | name_matches['recording_length_ms'].isna() | (duration_gap <= duration_limit)
name_matches = name_matches[duration_ok].copy()
name_matches['match_method'] = 'name_artist_duration'
name_matches['match_confidence'] = 0.8

real = (pd.concat([mbid_matches, name_matches], ignore_index=True)
    .sort_values(['listen_row_id', 'match_confidence'], ascending=[True, False])
    .drop_duplicates('listen_row_id', keep='first'))
real['duration_ms'] = real['duration_ms'].fillna(real['recording_length_ms'])
real = real.dropna(subset=['user_id', 'recording_id', 'timestamp', 'duration_ms'])
real['recording_id'] = real['recording_id'].astype(int)

unmatched = raw_listen_count - len(real)
print(f"Matched {len(real):,} listens with MusicBrainz catalog")
print(f"  MBID matches: {len(mbid_matches):,}")
print(f"  title+artist fallback matches: {len(name_matches):,}")
print(f"  unmatched listens skipped for recommender: {unmatched:,}")

# Derive the time-of-day bucket used by playlist features.
real['timestamp'] = pd.to_datetime(real['timestamp'], unit='s', errors='coerce')
real = real.dropna(subset=['timestamp'])
real['time_of_day'] = real['timestamp'].dt.hour.map(
    lambda h: 'morning' if h < 12 else 'afternoon' if h < 18 else 'night'
)

real['completed'] = real['duration_ms'] > 120_000

history = real[['user_id','recording_id','timestamp','time_of_day','duration_ms','completed']].copy()
history = history.sort_values('timestamp').reset_index(drop=True)
print(f"history → {len(history):,} rows | {history['user_id'].nunique()} users")

# 2. Build user profiles from the selected ListenBrainz users.
if 'user_name' in user_lookup.columns:
    usernames = (user_lookup.set_index('user_id')
        .reindex(range(1, N_USERS + 1))['user_name']
        .fillna(pd.Series([f'user_{i}' for i in range(1, N_USERS + 1)], index=range(1, N_USERS + 1)))
        .reset_index(drop=True))
else:
    usernames = [f'user_{i}' for i in range(1, N_USERS+1)]

users = pd.DataFrame({
    'user_id': range(1, N_USERS+1),
    'username': usernames,
    'age':     np.random.randint(18, 45, N_USERS),
    'country': np.random.choice(['PT','BR','US','UK','ES'], N_USERS)
})

# 3. Build a friend graph biased by shared artist taste.

# Use a power-law distribution so a small set of artists dominates listens.
artist_plays = (history
    .merge(recording[['id','artist_credit']], left_on='recording_id', right_on='id', how='left')
    .groupby('artist_credit').size()
    .reset_index(name='plays')
    .sort_values('plays', ascending=False))

# Estimate each user's top artists from generated listening history.
user_top_artists = (history
    .merge(recording[['id','artist_credit']], left_on='recording_id', right_on='id', how='left')
    .groupby(['user_id','artist_credit']).size()
    .reset_index(name='plays')
    .sort_values(['user_id','plays'], ascending=[True,False])
    .groupby('user_id').head(5))

user_artist_map = user_top_artists.groupby('user_id')['artist_credit'].apply(set).to_dict()

friends = []
friend_set = set()
uid_list = list(range(1, N_USERS+1))

for uid in uid_list:
    n_friends = random.randint(3, 15)
    pool = [u for u in uid_list if u != uid]

    # Rank friend candidates by overlapping top artists.
    shared = []
    my_artists = user_artist_map.get(uid, set())
    for cand in pool:
        overlap = len(my_artists & user_artist_map.get(cand, set()))
        shared.append((cand, overlap + 0.1))

    cands, weights = zip(*shared)
    weights = np.array(weights, dtype=float)
    weights /= weights.sum()

    chosen = np.random.choice(cands, size=min(n_friends, len(cands)),
                               replace=False, p=weights)
    for fid in chosen:
        if (fid, uid) not in friend_set:
            friend_set.add((uid, fid))
            friends.append({'user_id': uid, 'friend_id': fid})

friends = pd.DataFrame(friends)

# 4. Generate favourite artists with a global-popularity bias.

# Weight artist selection by global play rank.
n_artists   = len(artist_ids_list)
zipf_ranks  = np.arange(1, n_artists+1, dtype=float)
zipf_weights = 1.0 / zipf_ranks
zipf_weights /= zipf_weights.sum()

# Sort artists by play count before applying the weighted sample.
popular_artist_ids = (artist_plays
    .merge(artist_credit_name[['artist_credit','artist']].drop_duplicates(),
           on='artist_credit', how='left')
    .dropna(subset=['artist'])['artist']
    .astype(int).tolist())

# Fill any remaining slots with random artists.
if len(popular_artist_ids) < n_artists:
    missing = [a for a in artist_ids_list if a not in set(popular_artist_ids)]
    popular_artist_ids += missing
popular_artist_ids = popular_artist_ids[:n_artists]

fav_artists = []
for uid in users['user_id']:
    n_fav = random.randint(2, 8)
    chosen = np.random.choice(popular_artist_ids,
                               size=min(n_fav, len(popular_artist_ids)),
                               replace=False, p=zipf_weights[:len(popular_artist_ids)] /
                                               zipf_weights[:len(popular_artist_ids)].sum())
    for aid in chosen:
        fav_artists.append({'user_id': uid, 'artist_id': int(aid)})

fav_artists = pd.DataFrame(fav_artists)

# 5. Derive seed streaming events from listening history.
events = []
for _, row in history.iterrows():
    events.append({
        'event_type':   'play',
        'user_id':      row['user_id'],
        'recording_id': row['recording_id'],
        'ts':           row['timestamp'].isoformat(),
        'duration_ms':  row['duration_ms']
    })
    if not row['completed']:
        events.append({
            'event_type':   'skip',
            'user_id':      row['user_id'],
            'recording_id': row['recording_id'],
            'ts':           row['timestamp'].isoformat(),
            'position_ms':  random.randint(5000, 60000)
        })
events_df = pd.DataFrame(events)

# 6. Generate release notifications from favourite artists.
notifications = []
for _, row in fav_artists.iterrows():
    artist_releases = release[release['artist_credit'].isin(
        artist_credit_name[artist_credit_name['artist'] == row['artist_id']]['artist_credit']
    )]
    for _, rel in artist_releases.iterrows():
        notifications.append({
            'user_id':     row['user_id'],
            'artist_id':   row['artist_id'],
            'release_id':  rel['id'],
            'release_name': rel['name'],
            'notified_at': (datetime.now() - timedelta(days=random.randint(0,30))).isoformat()
        })
notifications = pd.DataFrame(notifications)

# Save generated TSV outputs.
users.to_csv(         f'{out}users.tsv',              sep='\t', index=False)
friends.to_csv(       f'{out}friends.tsv',            sep='\t', index=False)
fav_artists.to_csv(   f'{out}fav_artists.tsv',        sep='\t', index=False)
real[['listen_row_id','user_id','recording_id','match_method','match_confidence']].to_csv(
    f'{out}history_match_report.tsv', sep='\t', index=False)
history.to_csv(       f'{out}listening_history.tsv',  sep='\t', index=False)
events_df.to_csv(     f'{out}streaming_events.tsv',   sep='\t', index=False)
notifications.to_csv( f'{out}notifications.tsv',      sep='\t', index=False)

# Excel has a 1,048,576 row limit. Keep TSV files complete; Excel is preview only.
EXCEL_MAX_ROWS = 1_000_000
with pd.ExcelWriter(f'{out}synthetic_data.xlsx') as writer:
    users.head(EXCEL_MAX_ROWS).to_excel(        writer, sheet_name='users',         index=False)
    friends.head(EXCEL_MAX_ROWS).to_excel(      writer, sheet_name='friends',       index=False)
    fav_artists.head(EXCEL_MAX_ROWS).to_excel(  writer, sheet_name='fav_artists',   index=False)
    history.head(EXCEL_MAX_ROWS).to_excel(      writer, sheet_name='history',       index=False)
    events_df.head(EXCEL_MAX_ROWS).to_excel(    writer, sheet_name='streaming',     index=False)
    notifications.head(EXCEL_MAX_ROWS).to_excel(writer, sheet_name='notifications', index=False)

for name, df in [('users',users),('friends',friends),('fav_artists',fav_artists),
                 ('history',history),('streaming_events',events_df),
                 ('notifications',notifications)]:
    print(f"{name:25s} → {len(df)} rows")

Loading real listens...
Matched 497,329 listens with MusicBrainz catalog
  MBID matches: 18,512
  title+artist fallback matches: 480,503
  unmatched listens skipped for recommender: 3,246,043
history → 497,329 rows | 922 users
users                     → 1000 rows
friends                   → 8968 rows
fav_artists               → 5032 rows
history                   → 497329 rows
streaming_events          → 516949 rows
notifications             → 1468389 rows
